# Seismic Bumps Model Trainer
Run each cell in order. Upload your `seismic-bumpsdata.arff` file when prompted.

In [ ]:
!pip install scipy

In [3]:

import os
from google.colab import files

os.makedirs('./data', exist_ok=True)
print('Please upload your seismic-bumpsdata.arff file:')
uploaded = files.upload()

for filename in uploaded:
    os.rename(filename, f'./data/{filename}')
    print(f'Moved {filename} to ./data/')

Please upload your seismic-bumpsdata.arff file:


Saving seismic-bumpsdata.arff to seismic-bumpsdata.arff
Moved seismic-bumpsdata.arff to ./data/


In [10]:
import numpy as np
import pandas as pd
from scipy.io import arff
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

def plot(pandaframe):
    pandaframe.hist(bins=50, figsize=(20,15))
    plt.savefig("attribute_histogram_plots")
    plt.show()

def getBursts(pandaframe):
    rockbursts = []
    count = 0

    for index, bclass in enumerate(pandaframe['class']):
        if bclass == b'1':
            rockbursts.append(index)
    for index in rockbursts:
        print(pandaframe.iloc[index])
        count = count + 1

    print(count)

def prepareData(pandaframe):
    cat_to_num(pandaframe)
    xtrain, x_test, y_train, y_test = split_train_test(pandaframe)
    return normalizedata(xtrain, x_test, y_train, y_test)

# this is done right before training
def normalizedata(x_train, x_test, y_train, y_test):
    skewed = ['genergy', 'gpuls', 'energy', 'maxenergy']
    # Replace 0s with a small epsilon to avoid log(0) which results in -inf
    epsilon = np.finfo(float).eps
    for col in skewed:
        x_train[col] = np.log(x_train[col].replace(0, epsilon))
        x_test[col] = np.log(x_test[col].replace(0, epsilon))

    numericals = ['seismic', 'seismoacoustic', 'ghazard', 'genergy', 'gpuls', 'gdenergy', 'gdpuls', 'ghazard',
                  'nbumps', 'nbumps2', 'nbumps3', 'nbumps4', 'nbumps5', 'nbumps6', 'nbumps7', 'nbumps89', 'energy',
                  'maxenergy']
    normalizer = ColumnTransformer(transformers=[
        ('num', RobustScaler(), numericals),
        ('catagoric', OneHotEncoder(sparse_output=False), ['shift'])
    ])

    x_train_norm = normalizer.fit_transform(x_train)
    x_test_norm = normalizer.transform(x_test)

    return x_train_norm, x_test_norm, y_train.to_numpy(), y_test.to_numpy()

def split_train_test(df):
    x = df.drop('class', axis=1)
    y = df['class']

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    return x_train, x_test, y_train, y_test

def cat_to_num(df):
    df['class'] = df['class'].map({b'1': 1, b'0': 0})

    hazard_map = {b'a': 0, b'b': 1, b'c': 2, b'd': 3}
    df['seismic'] = df['seismic'].map(hazard_map)
    df['seismoacoustic'] = df['seismoacoustic'].map(hazard_map)
    df['ghazard'] = df['ghazard'].map(hazard_map)
    df['shift'] = df['shift'].map({b'W': 0, b'N': 1})

def compute_weights(y_train):
    weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
    class_weight_dict = {0: weights[0], 1: weights[1]}
    return class_weight_dict

arff_file = arff.loadarff('./data/seismic-bumpsdata.arff')

pandaframe = pd.DataFrame(arff_file[0])

#rockframe = pandaframe[pandaframe['class'] == b'1']
#plot(rockframe)

x_train, x_test, y_train, y_test = prepareData(pandaframe)
#print(x_test)

    #print(pandaframe)

    #getBursts(pandaframe)
#plot(pandaframe)


In [13]:
import tensorflow as tf
from tensorflow import keras

def main():
    arff_file = arff.loadarff('./data/seismic-bumpsdata.arff')
    pandaframe = pd.DataFrame(arff_file[0])
    x_train, x_test, y_train, y_test = prepareData(pandaframe)
    class_weights_dict = compute_weights(y_train)

    model = keras.models.Sequential()

    model.add(keras.layers.Input(shape=(x_train.shape[1],)))

    model.add(keras.layers.Dense(32, activation='relu'))
    model.add(keras.layers.BatchNormalization())
    model.add(keras.layers.Dropout(0.4))


    model.add(keras.layers.Dense(16, activation='relu'))
    model.add(keras.layers.BatchNormalization())
    model.add(keras.layers.Dropout(0.4))


    model.add(keras.layers.Dense(1, activation='sigmoid'))

    early_stop = keras.callbacks.EarlyStopping(
    monitor='val_recall',
    patience=10,
    restore_best_weights=True,
    mode='max'
    )

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.AUC(name='auc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall')
        ]
    )

    history = model.fit(
        x_train,
        y_train,
        epochs=25,
        batch_size=32,
        validation_data=(x_test, y_test),
        class_weight=class_weights_dict,
        callbacks=[early_stop]
    )

    model.evaluate(x_test, y_test)
    model.save("canaryv1.keras")


main()

Epoch 1/25
65/65 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.4475 - auc: 0.5137 - loss: 1.0358 - precision: 0.0701 - recall: 0.6029 - val_accuracy: 0.4043 - val_auc: 0.6970 - val_loss: 0.7705 - val_precision: 0.0873 - val_recall: 0.8529
Epoch 2/25
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.4790 - auc: 0.5857 - loss: 0.9214 - precision: 0.0810 - recall: 0.6691 - val_accuracy: 0.4468 - val_auc: 0.7396 - val_loss: 0.7901 - val_precision: 0.1013 - val_recall: 0.9412
Epoch 3/25
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5162 - auc: 0.6587 - loss: 0.7583 - precision: 0.0894 - recall: 0.6912 - val_accuracy: 0.4836 - val_auc: 0.7458 - val_loss: 0.7824 - val_precision: 0.1077 - val_recall: 0.9412
Epoch 4/25
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5210 - auc: 0.6612 - loss: 0.7621 - precision: 0.0902 - recall: 0.6912 - val_accuracy: 0.5300 - val_auc: 0.7436 - val_loss: 0.7515 - val_precision: 0.0965 - val_recall: 0.7353
Epoch 5/25
65/65 ━━━━━━━━━━━━━━━━━━━━ 0

NameError: name 'model' is not defined